In this tutorial/guide, we will be exploring the RAG (Retrievel Augmental Generation) technique with Claude (this content is derived from Anthropic skilljar site)

Imagine if we have a 1000 page financial document and want to ask Claude specific questions such as "What risk factors does this company have?". We will somehow have to extract the relevant info from the document to Claude but there are limits to how much text we can include in the prompt and a 1000-page is just too much!

A good soltion is to break the document into smaller chunks during a preprocessing step.
Then user asks a question, you find the chunks most relevant to their question and only include those in the prompts. RAG uses this approach.

The benefits of RAG are:
- Claude can focus on only the most relevant content
- Scales up to very large documents
- Works with multiple documents
- Smaller prompts cost less and run faster

The challenges with RAG are:
- Requires a preprocessing step to chunk the documents
- Need a search mechanism to find "relevant" chunks
- Included chunks might not contain all the context Claude needs
- How do we choose the best approach to chunk the text? (there are many ways to do this)

The most common approach for finding relevant chunks is using semantic search. Unlike keyword-based search that looks for exact word matches, semantic search uses text embeddings to understand the meaning and context of both the user's question and each text chunk.

Another approach is using text embedding. This is a numerical representation of the meaning contained in some text. Think of it as converting words and sentences into a format that computers can work with mathematically.
Here's how the process works:
- You feed text into an embedding model
- The model outputs a long list of numbers (the embedding)
- Each number ranges from -1 to +1
- These numbers represent different qualities or features of the input text

Each number is an embedding is essentially a "score" for some quality of the input text. However, we don't know precisely what each number represents. The actual meaning of each number is learned by the model during training and isn't directly interpretable by humans.

We can use VoyageAI for Embeddings since Anthropic doesn't currently provide embedding generation. You'll need to:
- Sign up for a separate VoyageAI
- Get an API key
- Add the key to your .env file

In [ ]:
VOYAGE_API_KEY="your_key_here"


In [ ]:
%pip install voyageai

In [ ]:
from dotenv import load_dotenv
import voyageai

load_dotenv()
client = voyageai.Client()

def generate_embedding(text, model="voyage-3-large", input_type="query"):
    result = client.embed([text], model=model, input_type=input_type)
    return result.embeddings[0]

When you run the `generate_embedding` function it will return a list of floating-point numbers representing the embedding. So, how do we use these embeddings effectively in our RAG pipeline for finding the most relevant content?

The full complete RAG flow is as follows:

1. Chunk Source Text
- We take the source document and break into manageable chunks

2. Generate Embeddings
- We convert each text chunk into numerical embeddings using an embedding model

3. Normalization
- We can use an embedding API to perform this step. This scales each vector to have a magnitude of 1.0 to make sure the embedding numbers are in a common scale/length so that they can be compared consistently

4. Store in Vector Database
- We store these embeddings in a vector database which is a specialized database optimized for storing, comparing and searching through long lists of numbers

4. Process User Query
- We run user query through the same embedding model and convert them into something that looks like `[0.1, 0.89]`

5. Find Similar Embeddings
- We then send the user's query embedding to our vector database and ask it to find the most similar stored embeddings

***Note that the vector database uses something called cosine similarity to determine which embeddings are most similar. It measures the cosine of the angle between the 2 vectors***

6. Create the final prompt
- We take the user's question/prompt and the most relevant text chunk we found and combine them into a final prompt and send it to Claude for a response

And that's how a full RAG pipeline works!
The following is the code.

In [ ]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)
chunks[2]  # Test to see the table of contents

embeddings = generate_embedding(chunks)

store = VectorIndex()

for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

user_embedding = generate_embedding("What did the software engineering dept do last year?")

results = store.search(user_embedding, 2)

for doc, distance in results:
    print(distance, "\n", doc["content"][0:200], "\n")


However, semantic search alone doesn't always return the best results. Sometimes you need exact term matches that semantic search might miss. The solution is a technique called BM25 that combines semantic search with lexical search.

BM25 (Best Match 25) is a popular algorithm for lexical search in RAG systems. Here's how it processes a search query:

1. Tokenize the query. Break user's question into individual terms. For example, "a INC-2023-Q4-011" becomes ["a", "INC-2023-Q4-011"].

2. Count term frequency. See how often each term appears across all the documents

3. Weight terms by importance. Terms that appear less frequently get higher important scores

4. Find best matches. Return the documents that contain more instances of the higher weighted terms

The following is the code to set up a basic BM25 search system.

In [ ]:
# 1. Chunk your text by sections
chunks = chunk_by_section(text)

# 2. Create a BM25 store and add documents
store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

# 3. Search the store
results = store.search("What happened with INC-2023-Q4-011?", 3)

# Print results
for doc, distance in results:
    print(distance, "\n", doc["content"][:200], "\n----\n")

In general, BM25 search > Semantic Search alone.

So, how do we combine BM25 and semantic search into a unified search pipeline?

Both the VectorIndex and BM25Index classes share nearly identical APIs. They both have `add_document()` and `search()` methods. This consistency makes it easy to wrap them in a new class called Reriever.

Retriever acts as a coordintor that forwards user queries to both indexes, collects their results, and merges them using a technique called reciprocal rank fusion.

What is reciprocal rank fusion?
Merging results from different search methods isn't as simple as concatenating lists. Each method uses different scoring systems, so we need a way to normalize and combine their rankings fairly. This is where reciprocal rank fusion comes in.
An example:

we search for information about "INC-2023-Q4-011" and get these results:

VectorIndex returns: Section 2 (rank 1), Section 7 (rank 2), Section 6 (rank 3)
BM25Index returns: Section 6 (rank 1), Section 2 (rank 2), Section 7 (rank 3)

We combine these into a single table showing each text chunk's rank from both indexes, then apply the RRF formula:

In [ ]:
RRF_score(d) = Σ(1 / (k + rank_i(d)))


Where k is a constant (often 60, but we'll use 1 for clearer results) and rank_i(d) is the rank of document d in the i-th ranking.

- Section 2: 1.0/(1+1) + 1.0/(1+2) = 0.833
- Section 7: 1.0/(1+2) + 1.0/(1+3) = 0.583
- Section 6: 1.0/(1+3) + 1.0/(1+1) = 0.75

The final ranking Section 2 beats the rest so it rises to the top

This is the implemented Retriever class that wraps multiple search indexes and provides a unified interface:

In [ ]:
class Retriever:
    def __init__(self, *indexes: SearchIndex):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)

    def add_document(self, document: Dict[str, Any]):
        for index in self._indexes:
            index.add_document(document)

    def search(self, query_text: str, k: int = 1, k_rrf: int = 60):
        # Get results from all indexes
        all_results = []
        for idx, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                # Track document ranks across indexes
                # Apply RRF scoring formula
        # Return merged and sorted results